# 01 Token Estimator

## Cell 1. English text

English text Notebook English text `TokenEstimator` English text, English text, English text tokenizer English text, English text.

In [1]:
from pathlib import Path
import sys
import time

import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").exists() and (candidate / "requirements.txt").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current notebook path.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.core.token_estimator import TokenEstimator

estimator = TokenEstimator()

LONG_ENGLISH_PARAGRAPH = (
    "Prompt optimization matters because every token influences latency, cost, and model focus. "
    "When a prompt includes repeated instructions, unnecessary qualifiers, or context that does not help the task, "
    "the model spends part of its limited attention budget on noise instead of the important constraints. "
    "A practical workflow is to keep the task, output format, and must-have details while trimming redundant wording, "
    "overlapping examples, and decorative language. Teams that measure token usage can compare drafts objectively, "
    "spot regressions early, and decide when a compact prompt is good enough. This matters even more in evaluation "
    "pipelines, multi-agent systems, and local inference setups where speed and throughput directly affect the user experience."
)

TEST_CASES = [
    ("english_short", "Hello, world!"),
    ("chinese_short", "English text,English text！"),
    ("mixed_language", "English text Python function English text JSON data"),
    ("english_long_paragraph", LONG_ENGLISH_PARAGRAPH),
    ("empty_string", ""),
]

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded {len(TEST_CASES)} token test cases.")

Project root: /Users/corey/Desktop/JHU/Generative AI/TokenLess
Loaded 5 token test cases.


## Cell 2. English text

English text, English text, English text, English text `exact_count`,English text `tokens / char` English text.

In [2]:
exact_rows = []

for label, text in TEST_CASES:
    token_count = estimator.exact_count(text)
    char_count = len(text)
    token_char_ratio = round(token_count / char_count, 4) if char_count else 0.0
    exact_rows.append(
        {
            "case": label,
            "text": text,
            "char_count": char_count,
            "exact_tokens": token_count,
            "tokens_per_char": token_char_ratio,
        }
    )

exact_df = pd.DataFrame(exact_rows)
exact_df

,case,text,char_count,exact_tokens,tokens_per_char
0,english_short,"Hello, world!",13,4,0.3077
1,chinese_short,"English text,English text！",6,7,1.1667
2,mixed_language,English text Python function English text JSON data,36,13,0.3611
3,english_long_paragraph,Prompt optimization matters because every toke...,757,130,0.1717
4,empty_string,,0,0,0.0000


## Cell 3. English text vs English text

English text `exact_count` English text `quick_estimate`,English text.

In [3]:
comparison_rows = []

for label, text in TEST_CASES:
    exact_tokens = estimator.exact_count(text)
    quick_tokens = estimator.quick_estimate(text)
    absolute_error = quick_tokens - exact_tokens
    error_rate_percent = round((abs(absolute_error) / exact_tokens) * 100, 2) if exact_tokens else 0.0
    comparison_rows.append(
        {
            "case": label,
            "exact_tokens": exact_tokens,
            "quick_estimate": quick_tokens,
            "absolute_error": absolute_error,
            "error_rate_percent": error_rate_percent,
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
print(f"Average error rate: {comparison_df['error_rate_percent'].mean():.2f}%")
comparison_df

Average error rate: 35.65%


,case,exact_tokens,quick_estimate,absolute_error,error_rate_percent
0,english_short,4,2,-2,50.00
1,chinese_short,7,10,3,42.86
2,mixed_language,13,22,9,69.23
3,english_long_paragraph,130,109,-21,16.15
4,empty_string,0,0,0,0.00


## Cell 4. English text tokenizer English text

English text `gpt-4o`, `gpt-3.5-turbo` English text,English text fallback English text.

In [4]:
sample_text = (
    "Tokenization differences matter when the same prompt mixes English, numbers like 2026, "
    "and Chinese such as English text in a single instruction."
)

model_rows = []
for model_name in ["gpt-4o", "gpt-3.5-turbo", "unknown-model-xyz"]:
    model_rows.append(
        {
            "model": model_name,
            "token_count": estimator.exact_count(sample_text, model=model_name),
        }
    )

tokenizer_df = pd.DataFrame(model_rows)
default_count = estimator.exact_count(sample_text)
fallback_count = tokenizer_df.loc[tokenizer_df["model"] == "unknown-model-xyz", "token_count"].iloc[0]

print(f"Default encoding token count: {default_count}")
print(f"Unknown model falls back to default encoding: {fallback_count == default_count}")
tokenizer_df

Default encoding token count: 32
Unknown model falls back to default encoding: True


,model,token_count
0,gpt-4o,29
1,gpt-3.5-turbo,32
2,unknown-model-xyz,32


## Cell 5. `compare_report` English text

English text prompt,English text `saved_tokens` English text `reduction_percent` English text.

In [5]:
original_prompt = (
    "You are an expert Python engineer. Please carefully analyze the following JSON payload, "
    "identify malformed fields, explain every issue in detail, and return a concise remediation checklist with examples."
)
optimized_prompt = (
    "Analyze this JSON payload, list malformed fields, and return a concise remediation checklist with examples."
)

report = estimator.compare_report(original_prompt, optimized_prompt, model="gpt-4o")
expected_saved_tokens = report.original_tokens - report.optimized_tokens
expected_reduction_percent = (
    round((expected_saved_tokens / report.original_tokens) * 100, 2)
    if report.original_tokens
    else 0.0
)

report_df = pd.DataFrame(
    [
        {
            **report.model_dump(),
            "saved_tokens_check": report.saved_tokens == expected_saved_tokens,
            "reduction_percent_check": round(report.reduction_percent, 2) == expected_reduction_percent,
        }
    ]
)

report_df

,original_tokens,optimized_tokens,saved_tokens,reduction_percent,saved_tokens_check,reduction_percent_check
0,34,18,16,47.058824,True,True


## Cell 6. English text

English text 1000+ English text 1000 English text `exact_count` English text `quick_estimate`,English text.

In [6]:
benchmark_paragraph = (
    "A realistic benchmark should use text that resembles production prompts with instructions, context, constraints, and output formatting. "
    "That matters because short strings can hide differences between exact tokenization and heuristic estimation. "
    "By repeating a moderately dense paragraph, we get a longer input that better reflects notebook experiments, evaluation pipelines, and prompt optimization workflows."
)
benchmark_text = " ".join([benchmark_paragraph] * 8)
iterations = 1000

exact_started = time.perf_counter()
for _ in range(iterations):
    estimator.exact_count(benchmark_text)
exact_elapsed = time.perf_counter() - exact_started

quick_started = time.perf_counter()
for _ in range(iterations):
    estimator.quick_estimate(benchmark_text)
quick_elapsed = time.perf_counter() - quick_started

benchmark_df = pd.DataFrame(
    [
        {
            "method": "exact_count",
            "iterations": iterations,
            "char_count": len(benchmark_text),
            "total_seconds": round(exact_elapsed, 4),
            "avg_ms_per_call": round((exact_elapsed / iterations) * 1000, 4),
        },
        {
            "method": "quick_estimate",
            "iterations": iterations,
            "char_count": len(benchmark_text),
            "total_seconds": round(quick_elapsed, 4),
            "avg_ms_per_call": round((quick_elapsed / iterations) * 1000, 4),
        },
    ]
)

faster_method = "exact_count" if exact_elapsed < quick_elapsed else "quick_estimate"
speedup = max(exact_elapsed, quick_elapsed) / min(exact_elapsed, quick_elapsed)
print(f"Faster method: {faster_method} ({speedup:.2f}x relative speedup)")
benchmark_df

Faster method: exact_count (6.10x relative speedup)


,method,iterations,char_count,total_seconds,avg_ms_per_call
0,exact_count,1000,3279,0.2028,0.2028
1,quick_estimate,1000,3279,1.2364,1.2364
